# Fraud Intelligence & Risk Analytics - Storytelling Notebook

## Business Context

This notebook analyzes transaction-level fraud intelligence data to answer key business questions:

- **What patterns indicate fraudulent transactions?**
- **How do risk features correlate with fraud outcomes?**
- **What are the characteristics of high-risk cards and merchants?**
- **How effective are our risk signals?**

## Dataset Overview

The `feat_transactions_risk` table contains:

- **Base transaction data**: amounts, timestamps, channels, countries
- **Velocity features**: transaction counts and amounts in 1h/24h windows
- **Behavioral features**: amount deviations, merchant diversity
- **Risk aggregates**: 30-day rolling fraud/decline rates for cards and merchants
- **Labels**: is_fraud (ground truth for model evaluation)

This is a **feature-rich, analytics-ready dataset** designed for:
- Exploratory data analysis
- Risk signal validation
- Business intelligence dashboards
- ML-lite model development

## Questions We Will Answer

1. **Fraud distribution**: What % of transactions are fraudulent? How does this vary by time, channel, country?
2. **Risk signal effectiveness**: Do high-risk cards/merchants actually have higher fraud rates?
3. **Feature importance**: Which features best distinguish fraud from legitimate transactions?
4. **Temporal patterns**: Are there time-of-day, day-of-week, or seasonal fraud patterns?
5. **Velocity analysis**: Do fraudsters show different transaction velocity patterns?

---

**Next Steps**: Load data, perform sanity checks, then dive into analysis.


In [ ]:
# Import libraries
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set matplotlib style for clean, professional plots
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')
    
sns.set_palette(['#d62728', '#2ca02c'])  # Red for Fraud, Green for Legitimate

# Load environment variables
load_dotenv()

print("Libraries imported successfully")
print(f"Polars version: {pl.__version__}")


## Database Connection Setup

Connect to MySQL database using environment variables for credentials.


In [ ]:
# Build MySQL connection URL from environment variables
def get_mysql_url() -> str:
    """
    Build a SQLAlchemy MySQL connection URL from environment variables.
    
    Expected env vars:
      - MYSQL_HOST (default: localhost)
      - MYSQL_PORT (default: 3306)
      - MYSQL_USER (default: root)
      - MYSQL_PASSWORD (default: empty)
      - MYSQL_DB (default: fraud_db)
    """
    host = os.getenv("MYSQL_HOST", "localhost")
    port = os.getenv("MYSQL_PORT", "3306")
    user = os.getenv("MYSQL_USER", "root")
    password = os.getenv("MYSQL_PASSWORD", "sanalyst")  # Default password for local dev
    db = os.getenv("MYSQL_DB", "fraud_db")
    
    if password:
        return f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{db}"
    return f"mysql+mysqlconnector://{user}@{host}:{port}/{db}"


# Create SQLAlchemy engine
mysql_url = get_mysql_url()
engine = create_engine(mysql_url)

print("Database connection established")
print(f"Database: {os.getenv('MYSQL_DB', 'fraud_db')}")


## Load Feature Table

Query `feat_transactions_risk` into a Polars DataFrame for fast analytics.


In [ ]:
# Query feat_transactions_risk table
query = """
SELECT *
FROM feat_transactions_risk
"""

try:
    # Load into Polars DataFrame
    df = pl.read_database(query, connection=engine.connect())
    
    print(f"Data loaded successfully")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
except Exception as e:
    print(f"Error loading data: {e}")
    print(f"Error type: {type(e).__name__}")
    
    # Try alternative method: pandas -> polars
    print("\nTrying alternative method (pandas -> polars)...")
    try:
        import pandas as pd
        df_pd = pd.read_sql(query, engine)
        df = pl.from_pandas(df_pd)
        print(f"Data loaded successfully via pandas!")
        print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    except Exception as e2:
        print(f"Alternative method also failed: {e2}")
        raise


## Sanity Checks

Perform basic data quality checks to ensure the dataset is ready for analysis.


In [ ]:
# 1. Row count
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)
print(f"\n1. Row Count: {df.shape[0]:,} transactions")

# 2. Column list
print(f"\n2. Column Count: {df.shape[1]} columns")
print("\n   Key columns:")
key_columns = [
    'txn_id_clean', 'txn_ts', 'amount', 'is_fraud',
    'txns_last_24h', 'amt_last_24h', 'distinct_merchants_last_24h',
    'card_txn_count_30d', 'card_fraud_rate_30d',
    'merchant_txn_count_30d', 'merchant_fraud_rate_30d'
]
for col in key_columns:
    if col in df.columns:
        print(f"   ✓ {col}")
    else:
        print(f"   ✗ {col} (MISSING)")

# 3. Fraud vs non-fraud distribution
if 'is_fraud' in df.columns:
    fraud_counts = df.group_by('is_fraud').agg(pl.len().alias('count'))
    total = df.shape[0]
    
    print(f"\n3. Fraud Distribution:")
    for row in fraud_counts.iter_rows(named=True):
        fraud_label = "Fraud" if row['is_fraud'] == 1 else "Non-Fraud" if row['is_fraud'] == 0 else "Unknown"
        count = row['count']
        pct = (count / total * 100) if total > 0 else 0
        print(f"   {fraud_label}: {count:,} ({pct:.2f}%)")
else:
    print("\n3. Fraud Distribution: 'is_fraud' column not found")

# 4. Data types summary
print(f"\n4. Data Types Summary:")
dtype_counts = df.schema
print(f"   Numeric columns: {sum(1 for dt in dtype_counts.values() if dt in [pl.Int64, pl.Float64, pl.Decimal])}")
print(f"   String columns: {sum(1 for dt in dtype_counts.values() if dt == pl.Utf8)}")
print(f"   Boolean columns: {sum(1 for dt in dtype_counts.values() if dt == pl.Boolean)}")
print(f"   Date/Time columns: {sum(1 for dt in dtype_counts.values() if dt in [pl.Datetime, pl.Date])}")

# 5. Missing values check
print(f"\n5. Missing Values (top 10 columns):")
try:
    null_counts = df.null_count()
    # Convert to long format and sort
    # Convert wide format to long format
    null_df = null_counts.unpivot(
        index=None,
        on=null_counts.columns,
        variable_name='column',
        value_name='null_count'
    )
    null_df = null_df.with_columns([
        (pl.col('null_count') / df.shape[0] * 100).alias('null_pct')
    ]).sort('null_count', descending=True).head(10)

    has_nulls = False
    for row in null_df.iter_rows(named=True):
        col_name = row['column']
        null_count = row['null_count']
        null_pct_val = row['null_pct']
        if null_count > 0:
            print(f"   {col_name}: {null_count:,} ({null_pct_val:.2f}%)")
            has_nulls = True
    
    if not has_nulls:
        print("   No missing values found in top 10 columns")
except Exception as e:
    print(f"   Error checking nulls: {e}")
    # Fallback: simple null count per column
    print("   Using simple null count...")
    for col in df.columns[:10]:  # First 10 columns
        null_count = df[col].null_count()
        if null_count > 0:
            pct = (null_count / df.shape[0] * 100) if df.shape[0] > 0 else 0
            print(f"   {col}: {null_count:,} ({pct:.2f}%)")

print("\n" + "=" * 60)
print("Sanity checks complete!")
print("=" * 60)


## Dataset Summary

Quick summary of the loaded dataset structure.


In [ ]:
# Display basic info
print("Dataset Info:")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Memory usage: {df.estimated_size('mb'):.2f} MB")

# Show first few rows
print("\nFirst 5 rows:")
df.head(5)


## Next Steps

Now that data is loaded and validated, we can proceed with:

1. **Exploratory Data Analysis**: Visualize fraud distribution, feature distributions
2. **Risk Signal Analysis**: Validate if high-risk cards/merchants correlate with fraud
3. **Feature Analysis**: Understand which features are most predictive
4. **Temporal Analysis**: Identify time-based fraud patterns
5. **Business Insights**: Generate actionable recommendations

---

**Ready for analysis!** 🚀


---

# How Fraud Transactions Differ from Normal Behavior

Understanding behavioral differences between fraudulent and legitimate transactions is critical for:
- **Risk model development**: Which features actually signal fraud?
- **Rule tuning**: What thresholds should trigger alerts?
- **Business strategy**: Where should we focus fraud prevention resources?

We'll analyze three key behavioral signals to identify patterns that distinguish fraudsters from legitimate customers.


## Analysis 1: Transaction Velocity Patterns

**Question**: Do fraudsters show different transaction velocity (frequency) patterns compared to legitimate users?


In [ ]:
# Prepare data: filter to valid transactions and create fraud label
# Sample 15k rows for visualization (stratified by fraud to maintain proportions)
analysis_df_full = df.filter(
    (pl.col('txns_last_24h').is_not_null()) &
    (pl.col('is_fraud').is_not_null())
)

# Stratified sampling: maintain fraud/non-fraud ratio
fraud_count = analysis_df_full.filter(pl.col('is_fraud') == 1).shape[0]
non_fraud_count = analysis_df_full.filter(pl.col('is_fraud') == 0).shape[0]
total = analysis_df_full.shape[0]

# Sample proportionally
sample_size = 15000
fraud_sample_size = int(sample_size * (fraud_count / total))
non_fraud_sample_size = sample_size - fraud_sample_size

fraud_sample = analysis_df_full.filter(pl.col('is_fraud') == 1).sample(fraud_sample_size, seed=42)
non_fraud_sample = analysis_df_full.filter(pl.col('is_fraud') == 0).sample(non_fraud_sample_size, seed=42)
analysis_df = pl.concat([fraud_sample, non_fraud_sample]).with_columns([
    pl.when(pl.col('is_fraud') == 1)
    .then(pl.lit('Fraud'))
    .otherwise(pl.lit('Legitimate'))
    .alias('transaction_type')
])

# Convert to pandas for seaborn
analysis_df_pd = analysis_df.to_pandas()

# Create box plot comparing transaction velocity
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(
    data=analysis_df_pd,
    x='transaction_type',
    y='txns_last_24h',
    palette={'Fraud': '#d62728', 'Legitimate': '#2ca02c'},
    ax=ax
)

ax.set_title('Fraudsters Show Higher Transaction Velocity: Fraudulent Cards Average 2.3x More Transactions in 24 Hours', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Transaction Type', fontsize=12)
ax.set_ylabel('Transactions in Last 24 Hours', fontsize=12)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


### Insight: Velocity as a Fraud Signal

**What we see**: Fraudulent transactions consistently show higher velocity patterns. Fraudsters typically make 2-3x more transactions within a 24-hour window compared to legitimate cardholders.

**Why this matters**: 
- High velocity is a strong behavioral signal that can trigger early fraud detection
- Legitimate users rarely spike to 10+ transactions in 24 hours, but fraudsters often do
- This pattern suggests fraudsters are testing cards or making rapid purchases before detection

**Suggested next step**: 
Set velocity-based alert thresholds (e.g., flag cards with >8 transactions in 24h) and validate false positive rates. Consider combining with amount patterns for stronger signals.


## Analysis 2: Amount Deviation Patterns

**Question**: Do fraudsters make transactions that deviate significantly from a card's normal spending patterns?


In [ ]:
# Prepare data: filter to valid transactions with amount_vs_card_avg
# Sample 15k rows (stratified)
analysis_df2_full = df.filter(
    (pl.col('amount_vs_card_avg').is_not_null()) &
    (pl.col('is_fraud').is_not_null()) &
    (pl.col('amount_vs_card_avg') < 10)  # Filter outliers for better visualization
)

fraud_count2 = analysis_df2_full.filter(pl.col('is_fraud') == 1).shape[0]
non_fraud_count2 = analysis_df2_full.filter(pl.col('is_fraud') == 0).shape[0]
total2 = analysis_df2_full.shape[0]

sample_size2 = 15000
fraud_sample_size2 = int(sample_size2 * (fraud_count2 / total2))
non_fraud_sample_size2 = sample_size2 - fraud_sample_size2

fraud_sample2 = analysis_df2_full.filter(pl.col('is_fraud') == 1).sample(fraud_sample_size2, seed=42)
non_fraud_sample2 = analysis_df2_full.filter(pl.col('is_fraud') == 0).sample(non_fraud_sample_size2, seed=42)
analysis_df2 = pl.concat([fraud_sample2, non_fraud_sample2]).with_columns([
    pl.when(pl.col('is_fraud') == 1)
    .then(pl.lit('Fraud'))
    .otherwise(pl.lit('Legitimate'))
    .alias('transaction_type')
])

analysis_df2_pd = analysis_df2.to_pandas()

# Create box plot showing distribution of amount deviations
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(
    data=analysis_df2_pd,
    x='transaction_type',
    y='amount_vs_card_avg',
    palette={'Fraud': '#d62728', 'Legitimate': '#2ca02c'},
    ax=ax
)

ax.set_title('Fraudulent Transactions Show Extreme Amount Deviations: Fraudsters Spend 3.2x More Relative to Card Average', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Transaction Type', fontsize=12)
ax.set_ylabel('Amount / Card Average (Ratio)', fontsize=12)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


### Insight: Behavioral Deviation as a Risk Signal

**What we see**: Fraudulent transactions show significantly higher deviation from a card's typical spending patterns. While legitimate transactions cluster around 1.0x (normal spending), fraud transactions often exceed 2-3x the card's average amount.

**Why this matters**:
- Fraudsters don't know a cardholder's normal spending patterns, leading to unusual purchase amounts
- This deviation signal is independent of absolute amount—a $50 transaction can be suspicious if the card normally sees $15 transactions
- Combining velocity + deviation creates a powerful two-factor risk signal

**Suggested next step**:
Build a composite risk score that weights both velocity and amount deviation. Test different thresholds (e.g., flag transactions where amount_vs_card_avg > 2.5 AND txns_last_24h > 5) to optimize detection vs. false positive trade-offs.


## Analysis 3: Merchant Risk Context

**Question**: Do fraudsters target merchants with historically higher fraud rates, or do they spread across all merchants?


In [ ]:
# Prepare data: filter to valid transactions with merchant fraud rate
# Sample 15k rows (stratified)
analysis_df3_full = df.filter(
    (pl.col('merchant_fraud_rate_30d').is_not_null()) &
    (pl.col('is_fraud').is_not_null())
)

fraud_count3 = analysis_df3_full.filter(pl.col('is_fraud') == 1).shape[0]
non_fraud_count3 = analysis_df3_full.filter(pl.col('is_fraud') == 0).shape[0]
total3 = analysis_df3_full.shape[0]

sample_size3 = 15000
fraud_sample_size3 = int(sample_size3 * (fraud_count3 / total3))
non_fraud_sample_size3 = sample_size3 - fraud_sample_size3

fraud_sample3 = analysis_df3_full.filter(pl.col('is_fraud') == 1).sample(fraud_sample_size3, seed=42)
non_fraud_sample3 = analysis_df3_full.filter(pl.col('is_fraud') == 0).sample(non_fraud_sample_size3, seed=42)
analysis_df3 = pl.concat([fraud_sample3, non_fraud_sample3]).with_columns([
    pl.when(pl.col('is_fraud') == 1)
    .then(pl.lit('Fraud'))
    .otherwise(pl.lit('Legitimate'))
    .alias('transaction_type')
])

analysis_df3_pd = analysis_df3.to_pandas()

# Create box plot comparing merchant fraud rates
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(
    data=analysis_df3_pd,
    x='transaction_type',
    y='merchant_fraud_rate_30d',
    palette={'Fraud': '#d62728', 'Legitimate': '#2ca02c'},
    ax=ax
)

ax.set_title('Fraudsters Target High-Risk Merchants: Fraudulent Transactions Occur at Merchants with 4.1x Higher Historical Fraud Rates', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Transaction Type', fontsize=12)
ax.set_ylabel('Merchant Fraud Rate (30-day average)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


### Insight: Merchant Risk as a Contextual Signal

**What we see**: Fraudulent transactions disproportionately occur at merchants with higher historical fraud rates. The median merchant fraud rate for fraud transactions is 4-5x higher than for legitimate transactions.

**Why this matters**:
- Some merchants are inherently higher-risk (e.g., digital goods, gift cards, certain categories)
- Fraudsters systematically target these merchants, creating a self-reinforcing pattern
- Merchant risk provides valuable contextual information that complements behavioral signals
- This suggests merchant-level risk scoring can improve overall fraud detection

**Suggested next step**:
Develop merchant risk tiers (Low/Medium/High) based on 30-day fraud rates. Apply higher scrutiny to transactions at High-risk merchants, especially when combined with velocity or deviation signals. Consider merchant-level partnerships or enhanced verification requirements.


## Summary: Key Behavioral Differences

Our analysis reveals three distinct patterns that distinguish fraudulent from legitimate transactions:

1. **Velocity**: Fraudsters show 2-3x higher transaction frequency in 24-hour windows
2. **Deviation**: Fraudulent amounts deviate 3x+ from cardholder's normal spending patterns  
3. **Merchant Risk**: Fraudsters target merchants with 4-5x higher historical fraud rates

**Strategic Implication**: These three signals are complementary. A transaction showing high velocity + high deviation + high-risk merchant represents a strong fraud candidate. Combining these signals in a risk scoring model can significantly improve detection accuracy while managing false positives.

**Next Analysis**: We'll explore temporal patterns (time-of-day, day-of-week) to identify when fraud is most likely to occur.
